# Данные МНСК в pandas

Запускайте ячейки сверху вниз в Jupyter на компьютере с папкой проекта. Можно сохранить этот ноутбук в любой другой папке. Требуется Python 3.10 или новее. Если пакеты ещё не установлены, выполните в отдельной ячейке:

```python
%pip install pandas numpy "openpyxl>=3.1,<3.2" matplotlib
```

Параметр `ROOT` указывает на локальную копию. Код читает книги с диска и создаёт объекты в памяти ядра Jupyter. Для загрузки достаточно выполнить первые две ячейки кода. Остальные показывают работу с конкретными листами МНСК.

In [41]:
from contextlib import ExitStack, closing
from pathlib import Path, PurePosixPath
from urllib.parse import unquote
from xml.etree import ElementTree as ET
from zipfile import ZipFile
import re

import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

ROOT = Path(r"C:\Users\chand\source\repos\Github\china-public-sector-performance")
MAIN = "МНСК.xlsx"

## 1. Загрузить все листы и сопоставить внешние ссылки

У каждого листа есть ключ `(путь книги относительно ROOT, имя листа)`. В `values` лежат значения, в `formulas` — содержимое ячеек с выражениями Excel на местах формул. Координата `B5` читается как `.at[5, "B"]`. Номера строк, буквы столбцов, заголовки, промежутки и примечания сохраняются, в том числе на скрытых листах. При объединённых ячейках значение находится в их верхней левой ячейке.

Excel хранит в XLSX текст формулы и может сохранить результат последнего вычисления. pandas использует этот результат как снимок на момент сохранения. Изменения исходных значений в DataFrame сами по себе формулы Excel не пересчитывают. Если нужен свежий результат всей книги, можно открыть отдельную копию проекта в Excel, обновить внешние связи на файлы этой копии, пересчитать, сохранить и указать её путь в `ROOT`.

`catalog` перечисляет листы. `links` связывает номера вида `[1]` в формулах каждой книги с исходными файлами. Для ссылки со старым абсолютным адресом проверяется совпадение папки и имени внутри проекта. `issues` показывает адреса ошибок Excel и формул без сохранённого результата. Пустая строка, которую вернула формула, остаётся пустой строкой. Выражения используют имена функций в записи XLSX, например `IFERROR`, даже если Excel отображает их по-русски.

In [42]:
def local_source(book, target, files):
    """Сопоставляем внешний адрес только с уже найденными внутри проекта книгами."""
    path = unquote(target).replace("\\", "/")
    relative = (ROOT / book).parent / path
    if not path.startswith("file:") and relative.resolve() in files:
        return files[relative.resolve()], "по адресу ссылки"
    parts = [x.casefold() for x in PurePosixPath(path).parts]
    # Для старого абсолютного пути требуем совпадения как минимум папки и имени.
    for size in range(len(parts), 1, -1):
        matches = [key for key in files.values()
                   if [x.casefold() for x in PurePosixPath(key).parts][-size:] == parts[-size:]]
        if len(matches) == 1:
            return matches[0], "по окончанию пути"
        if matches:
            return None, "неоднозначная ссылка"
    return None, "источник не найден"


def empty_strings(archive, sheet):
    # У Excel строковый результат "" записан как t="str" с пустым тегом <v>.
    # При data_only=True openpyxl возвращает для него None, как и при отсутствии кэша.
    with archive.open(sheet._worksheet_path) as stream:
        root = ET.parse(stream).getroot()
    result = set()
    for cell in root.iterfind(".//{*}c[@t='str']"):
        value = cell.find("{*}v")
        if value is not None and value.text is None:
            result.add(cell.attrib["r"])
    return result


def excel_frame(rows):
    df = pd.DataFrame(rows, dtype=object)
    df.index = pd.RangeIndex(1, len(df) + 1, name="строка Excel")
    df.columns = [get_column_letter(i) for i in range(1, df.shape[1] + 1)]
    return df


def read_project():
    if not ROOT.is_dir():
        raise FileNotFoundError(f"Проверьте путь ROOT: {ROOT}")
    books = sorted(p for p in ROOT.rglob("*") if p.is_file()
                   and p.suffix.lower() == ".xlsx" and not p.name.startswith("~$"))
    files = {p.resolve(): p.relative_to(ROOT).as_posix() for p in books}
    if MAIN not in files.values():
        raise FileNotFoundError(f"В корне проекта не найдена книга {MAIN}.")

    values, formulas = {}, {}
    sheets, links, issues = [], [], []
    for path in books:
        book = files[path.resolve()]
        try:
            with ExitStack() as stack:
                wb = stack.enter_context(closing(load_workbook(path, read_only=True)))
                cache = stack.enter_context(closing(load_workbook(
                    path, read_only=True, data_only=True, keep_links=False)))
                archive = stack.enter_context(ZipFile(path))
                for number, link in enumerate(wb._external_links, start=1):
                    target = link.file_link.Target
                    source, status = local_source(book, target, files)
                    links.append((book, number, target, source, status))

                for sheet in wb.worksheets:
                    key = (book, sheet.title)
                    saved = cache[sheet.title]
                    blanks = empty_strings(archive, sheet)
                    # Читаем фактические строки даже у файлов с неверным XML-размером листа.
                    sheet.reset_dimensions()
                    saved.reset_dimensions()
                    vrows, frows = [], []
                    formula_count = blank_count = 0
                    for left, right in zip(sheet.rows, saved.rows, strict=True):
                        vr, fr = [], []
                        for cell, cached in zip(left, right, strict=True):
                            value = cached.value
                            if value is None and getattr(cell, "coordinate", None) in blanks:
                                value = ""
                            expression = cell.value
                            if cell.data_type == "f":
                                formula_count += 1
                                # openpyxl разворачивает общие формулы в формулу каждой ячейки.
                                # Для особых формул массива/таблицы сохраняем также их атрибуты.
                                if not isinstance(expression, str):
                                    expression = dict(vars(expression), type=type(expression).__name__)
                                if value is None:
                                    issues.append((*key, cell.coordinate, "нет сохранённого результата"))
                                elif value == "":
                                    blank_count += 1
                            if cached.data_type == "e":
                                issues.append((*key, cell.coordinate, value))
                            vr.append(value)
                            fr.append(expression)
                        vrows.append(vr)
                        frows.append(fr)
                    values[key], formulas[key] = excel_frame(vrows), excel_frame(frows)
                    sheets.append((*key, *values[key].shape, formula_count, blank_count))
        except Exception as exc:
            raise RuntimeError(f"Не удалось полностью прочитать {book}: {exc}") from exc

    catalog = pd.DataFrame(sheets, columns=["книга", "лист", "строк", "столбцов", "формул", "формул с результатом «»"])
    links = pd.DataFrame(links, columns=["книга", "номер", "адрес Excel", "источник", "статус"])
    issues = pd.DataFrame(issues, columns=["книга", "лист", "ячейка", "проблема"])
    return values, formulas, catalog, links, issues


values, formulas, catalog, links, issues = read_project()
print(f"Книг: {catalog['книга'].nunique()}, листов: {len(catalog)}, формул: {catalog['формул'].sum():,}")
display(catalog.head(), links.groupby(["книга", "статус"]).size().rename("ссылок").to_frame())
print("Ячеек с ошибками Excel или без сохранённого результата формулы:", len(issues))
display(issues.head(10))

Книг: 75, листов: 298, формул: 204,698


,книга,лист,строк,столбцов,формул,формул с результатом «»
0,01_lighting/1.xlsx,Sheet1,36,26,0,0
1,01_lighting/areas.xlsx,Province Areas,35,5,0,0
2,01_lighting/areas.xlsx,Area by Year,39,26,0,0
3,01_lighting/population.xlsx,Sheet1,36,26,0,0
4,02_electricity/2.xlsx,Sheet1,36,26,0,0


ссылок
книга     статус                   
I.xlsx    по адресу ссылки        1
W.xlsx    по адресу ссылки        1
МНСК.xlsx по адресу ссылки       70
          по окончанию пути       1

Ячеек с ошибками Excel или без сохранённого результата формулы: 0


,книга,лист,ячейка,проблема


Чтобы получить данные одной провинции, выполните `panel.loc["Пекин"]`. Для среза по всем провинциям за один год используйте `panel.xs(2019, level="год")`. Результаты пространственного анализа по годам доступны как `values[MAIN, "Итог_Moran_Соседство"]`.

Структура и результаты проверены на [публичной версии проекта](https://github.com/Settara/china-public-sector-performance/tree/99c67e6db75be59972220a7900d8935ca2e2a08e). Локальный запуск использует файлы из `ROOT`.

## 5. Таблицы и карта в Gradio
Выполните ячейки загрузки и сборки панели выше. Для карты поместите папку `maps` с границами провинций и локальной подложкой Plotly в проект с МНСК. Источник контуров — [Natural Earth, 1:10m, Admin 1](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-1-states-provinces/). Контуры Тайваня, Гонконга и Макао служат географическими ориентирами.

```python
%pip install "gradio>=5.49,<6" "plotly>=5.24,<6"
```

In [ ]:
import re
from html import escape
from numbers import Integral, Real
from textwrap import wrap

import gradio as gr
import pandas as pd


# Список таблиц из уже загруженных книг
INDICATOR = "Показатель"
RAW = "Ненормированный"
NORMAL = "Нормированный"
ALL = "Все таблицы проекта"

groups = {
    "Индексы и агрегаты": [],
    "Таблицы Морана": [],
    "Исходные книги": [],
    "Другие листы МНСК": [],
    ALL: [],
}

def with_header(df):
    table = df.iloc[1:].copy()
    table.columns = df.iloc[0].tolist()
    return table


dictionary = with_header(values[MAIN, "Список показателей"])
numbers = pd.to_numeric(dictionary["№"], errors="coerce")
metadata = dictionary.loc[numbers.notna()].copy()
metadata["№"] = numbers.dropna().astype(int)
metadata = metadata.set_index("№")
if not metadata.index.is_unique:
    raise ValueError("В справочнике повторяются номера показателей.")

names = with_header(values[MAIN, "Adjacency_names"])[["Province", "Русский"]].dropna()
to_ru = dict(zip(names["Province"], names["Русский"]))
to_ru.update({name: name for name in names["Русский"]})
to_ru["Xizang"] = to_ru["Tibet"]


def annual_series(sheet_name):
    df = values[MAIN, sheet_name]
    headers = df.index[df["A"].eq("Region")]
    if len(headers) != 1:
        raise ValueError(f"Проверьте заголовок Region на листе {sheet_name}")
    header = headers[0]
    years = {c: int(v) for c, v in df.loc[header].items()
             if re.fullmatch(r"(19|20)\d{2}", str(v))}
    if not years or len(set(years.values())) != len(years):
        raise ValueError(f"Проверьте годы на листе {sheet_name}")
    # В проверенной книге после Region идёт непрерывный блок из 31 провинции.
    block = df.loc[header + 1:header + len(names)].copy()
    regions = block["A"].map(to_ru)
    if set(regions) != set(names["Русский"]) or regions.duplicated().any():
        raise ValueError(f"Проверьте список и написание провинций на листе {sheet_name}")
    wide = block[list(years)].rename(columns=years)
    wide.index = regions.rename("провинция")
    long = wide.reset_index().melt(id_vars="провинция", var_name="год", value_name="значение")
    long["год"] = long["год"].astype(int)
    # Аналитическая копия. В values сохраняются исходные тексты, пустые строки и ошибки Excel.
    long["значение"] = pd.to_numeric(long["значение"], errors="coerce")
    annual = long.set_index(["провинция", "год"])["значение"]
    if not annual.index.is_unique:
        raise ValueError(f"Повторяется пара провинция и год на листе {sheet_name}")
    return annual


block_choices = []
index_labels = {"I_Сводный": "9.I_Сводный"}
indicators_by_block = {}
for block_number, (block_name, rows) in enumerate(
    metadata.sort_index().groupby("Блок", sort=False), start=1
):
    index_labels[f"I_Блок{block_number}"] = f"{block_number}.I_{block_name}"
    block_choices.append((f"{block_number}.{block_name}", block_number))
    indicators_by_block[block_number] = [
        (f"{block_number}.{number}.{name}", number)
        for number, name in rows["Название"].items()
    ]

sheet_keys = sorted(values)
indicator_sheets = {}

for sheet_id, (book, sheet) in enumerate(sheet_keys):
    choice = (f"{book} › {sheet}", sheet_id)
    groups[ALL].append(choice)

    raw_match = re.match(r"^(\d+)\.", sheet)
    norm_match = re.match(r"^N_(\d+)_", sheet)

    if book == MAIN and (raw_match or norm_match):
        number = int((raw_match or norm_match).group(1))
        version = RAW if raw_match else NORMAL
        versions = indicator_sheets.setdefault(number, {})

        if version in versions:
            raise ValueError(f"Для показателя №{number} найдены два листа «{version}».")

        versions[version] = (book, sheet)

    elif book != MAIN:
        groups["Исходные книги"].append(choice)
    elif "moran" in sheet.casefold():
        groups["Таблицы Морана"].append(choice)
    elif sheet in index_labels:
        groups["Индексы и агрегаты"].append((index_labels[sheet], sheet_id))
    elif not sheet.startswith(("P_Блок", "P_Сводный")):
        groups["Другие листы МНСК"].append(choice)


def cell_text(value):
    """Форматирование только для экрана."""
    if pd.isna(value):
        return ""

    if isinstance(value, bool):
        return str(value)

    if isinstance(value, Integral):
        return str(value)

    if isinstance(value, Real):
        # Доли и малые разности сохраняют до семи значащих цифр.
        spec = ".7g" if 0 < abs(value) < 1 else ".2f"
        return format(value, spec).replace(".", ",")

    return escape(str(value))


def table_cell(value):
    text = cell_text(value)
    if isinstance(value, Real) and pd.notna(value) and not isinstance(value, (bool, Integral)):
        exact = str(value).replace(".", ",")
        return f'<td title="{escape(exact)}">{text}</td>'
    return f"<td>{text}</td>"


def sheet_html(key):
    book, sheet = key
    df = values[key]
    rows = []
    annual = False

    for row in df.itertuples(index=False, name=None):
        # На годовых листах МНСК заголовок начинается с Region,
        # а следующие столбцы содержат годы.
        year_header = (
            len(row) >= 2
            and str(row[0]).strip().casefold() == "region"
            and re.fullmatch(r"(?:19|20)\d{2}(?:\.0)?", str(row[1]).strip())
            is not None
        )

        if year_header:
            annual = True
            cells = []
            for value in row:
                text = str(value).strip()
                if re.fullmatch(r"(?:19|20)\d{2}(?:\.0)?", text):
                    text = text[:4]  # 2024 остаётся годом без ",00"
                else:
                    text = cell_text(value)
                cells.append(f"<th>{text}</th>")
        else:
            cells = [table_cell(value) for value in row]

        rows.append("<tr>" + "".join(cells) + "</tr>")

    # Фиксированная раскладка нужна для годовых таблиц.
    # У справочников и таблиц Морана ширина зависит от их содержимого.
    if annual:
        other_columns = df.shape[1] - 1
        table_style = f'class="annual" style="--other-cols:{other_columns}"'
        columns = (
            '<colgroup><col class="province-col">'
            f'<col span="{other_columns}"></colgroup>'
        )
    else:
        table_style = 'class="regular"'
        columns = ""

    title = escape(f"{book} › {sheet}")
    return (
        f'<p class="sheet-title">{title}</p>'
        '<div class="sheet-scroll" tabindex="0">'
        f'<table {table_style} aria-label="{title}">'
        f'{columns}<tbody>{"".join(rows)}</tbody>'
        '</table></div>'
    )


def select_table(category, block, number, version, selected, event: gr.EventData):
    is_indicator = category == INDICATOR
    indicators = indicators_by_block.get(block, [])
    if number not in {idx for _, idx in indicators}:
        number = None
    versions = indicator_sheets.get(number, {})
    options = [name for name in (RAW, NORMAL) if name in versions] or [RAW, NORMAL]
    if version not in options:
        version = options[0]

    choices = groups.get(category, [])
    if event.target is category_box or selected not in {idx for _, idx in choices}:
        selected = None

    if is_indicator and versions:
        html = sheet_html(versions[version])
    elif is_indicator:
        html = "Выберите показатель." if indicators else "Выберите блок."
    elif selected is None:
        html = "Выберите книгу и лист."
    else:
        html = sheet_html(sheet_keys[selected])

    return (
        gr.update(visible=is_indicator),
        gr.update(visible=is_indicator, interactive=bool(indicators), choices=indicators, value=number),
        gr.update(
            visible=is_indicator,
            interactive=bool(versions),
            choices=options,
            value=version,
        ),
        gr.update(visible=not is_indicator, choices=choices, value=selected),
        html,
    )


css = """
.gradio-container {
    width: 100% !important;
    max-width: none !important;
}

:is(#sheet-output, #province-output, #balance-output) {
    min-width: 0;
}

:is(#sheet-output, #province-output, #balance-output) .sheet-title {
    margin: 0 0 6px;
    font-weight: 600;
}

:is(#sheet-output, #province-output, #balance-output) .sheet-scroll {
    width: 100%;
    max-width: 100%;
    height: auto;
    max-height: none;
    overflow-x: auto;
    overflow-y: hidden;
    container-type: inline-size;
    border: 1px solid #bcc7d0;
}

:is(#sheet-output, #province-output, #balance-output) table {
    margin: 0;
    border-collapse: separate;
    border-spacing: 0;
    color: var(--body-text-color);
    font-size: 12px;
    line-height: 1.2;
}

:is(#sheet-output, #province-output, #balance-output) .annual {
    --province-width: clamp(110px, 13cqw, 175px);
    --year-width: clamp(64px, calc((100cqw - var(--province-width) - 2px) / 14), 98px);
    table-layout: fixed;
    width: calc(var(--province-width) + var(--other-cols) * var(--year-width));
    max-width: none;
}

:is(#sheet-output, #province-output, #balance-output) .annual .province-col {
    width: var(--province-width);
}

:is(#sheet-output, #province-output, #balance-output) .regular {
    width: max-content;
    min-width: 100%;
    max-width: none;
}

:is(#sheet-output, #province-output, #balance-output) td,
:is(#sheet-output, #province-output, #balance-output) th {
    box-sizing: border-box;
    border: solid #cdd5dc;
    border-width: 0 1px 1px 0;
    padding: 3px 5px;
    text-align: right;
    vertical-align: middle;
    font-variant-numeric: tabular-nums;
}

:is(#sheet-output, #province-output, #balance-output) .annual td,
:is(#sheet-output, #province-output, #balance-output) .annual th {
    white-space: normal;
    overflow-wrap: anywhere;
}

:is(#sheet-output, #province-output, #balance-output) .regular td,
:is(#sheet-output, #province-output, #balance-output) .regular th {
    white-space: pre-wrap;
    max-width: 36rem;
}

/* При прокрутке к ранним годам названия провинций остаются слева. */
:is(#sheet-output, #province-output, #balance-output) tr > :first-child {
    position: sticky;
    left: 0;
    z-index: 1;
    text-align: left;
    background: var(--background-fill-primary, white);
}

:is(#sheet-output, #province-output, #balance-output) th,
:is(#sheet-output, #province-output, #balance-output) tr > th:first-child {
    background: var(--background-fill-secondary, #edf2f5);
    font-weight: 600;
}

#province-output .sheet-scroll {
    max-height: 70vh;
    overflow: auto;
}
#balance-output table {
    width: 100%;
    min-width: 65rem;
    table-layout: fixed;
}
#balance-output thead th {
    text-align: center;
    overflow-wrap: anywhere;
}
#balance-output tr > :first-child {
    width: clamp(13rem, 26cqw, 22rem);
    overflow-wrap: anywhere;
}
#balance-output tfoot td {
    font-weight: 700;
    background: var(--background-fill-secondary, #edf2f5);
}
#province-output .annual {
    --province-width: clamp(145px, 29cqw, 360px);
}
#province-output thead th {
    position: sticky;
    top: 0;
    z-index: 2;
    background: var(--background-fill-secondary, #edf2f5);
}
#province-output thead th:first-child {
    z-index: 3;
}
"""


import json
from functools import lru_cache
import plotly.graph_objects as go

geo_file = ROOT / "maps" / "china_provinces.geojson"
with geo_file.open(encoding="utf-8") as stream:
    boundaries = json.load(stream)

# Локальная подложка Plotly позволяет открывать географический график без CDN.
with geo_file.with_name("world_110m.json").open(encoding="utf-8") as stream:
    world_json = json.dumps(json.load(stream), separators=(",", ":")).replace("<", r"\u003c")
map_head = ("<script>window.PlotlyGeoAssets = window.PlotlyGeoAssets || {topojson:{}};"
            "window.PlotlyGeoAssets.topojson.world_110m = " + world_json + ";</script>")

province_ids = [f["id"] for f in boundaries["features"] if f["properties"]["sample"]]
if len(province_ids) != 31 or set(province_ids) != set(names["Province"]):
    raise ValueError("Названия территорий GeoJSON расходятся со справочником Adjacency_names.")
ru_to_id = {to_ru[pid]: pid for pid in province_ids}

adj = values[MAIN, "Adjacency"]
header = adj.index[adj["A"].eq("Province")].item()
adj = adj.loc[header + 1:].set_axis(adj.loc[header], axis=1).set_index("Province")
adj = adj.loc[province_ids, province_ids]

RANK = "Место за выбранный период"
FIXED = "Значение индекса (1–100)"
overview = "I_Сводный"
map_options = [("Сводный индекс", overview)]
for n, block in enumerate(metadata["Блок"].drop_duplicates(), start=1):
    sheet = f"I_Блок{n}"
    if (MAIN, sheet) in values:
        map_options.append((f"Индекс блока {n} · {block}", sheet))
for number in sorted(indicator_sheets):
    map_options.append((f"{number}. {metadata.at[number, 'Название']}", str(number)))

@lru_cache(maxsize=160)
def province_years(sheet):
    return annual_series(sheet).unstack("год").sort_index(axis=1)

# Краткие китайские названия по английским ключам справочника Adjacency_names.
province_zh = {
    "Beijing": "北京", "Tianjin": "天津", "Hebei": "河北", "Shanxi": "山西",
    "Inner Mongolia": "内蒙古", "Liaoning": "辽宁", "Jilin": "吉林", "Heilongjiang": "黑龙江",
    "Shanghai": "上海", "Jiangsu": "江苏", "Zhejiang": "浙江", "Anhui": "安徽",
    "Fujian": "福建", "Jiangxi": "江西", "Shandong": "山东", "Henan": "河南",
    "Hubei": "湖北", "Hunan": "湖南", "Guangdong": "广东", "Guangxi": "广西",
    "Hainan": "海南", "Chongqing": "重庆", "Sichuan": "四川", "Guizhou": "贵州",
    "Yunnan": "云南", "Tibet": "西藏", "Shaanxi": "陕西", "Gansu": "甘肃",
    "Qinghai": "青海", "Ningxia": "宁夏", "Xinjiang": "新疆",
}
province_labels = {
    ru: f"{ru} · {en} · {province_zh[en]}"
    for en, ru in names.itertuples(index=False, name=None)
}
province_choices = [(label, ru) for ru, label in province_labels.items()]

# Объединяем ненормированные значения по названию провинции и году.
province_data = pd.concat({
    label: province_years(indicator_sheets[number][RAW][1])
    for choices in indicators_by_block.values()
    for label, number in choices
}, names=["Показатель", "провинция"]).sort_index(axis=1, ascending=False)


def province_table(province):
    if province is None:
        return "Выберите провинцию."
    table = province_data.xs(province, level="провинция")
    years = "".join(f'<th scope="col">{year}</th>' for year in table.columns)
    rows = []
    for label, *observations in table.itertuples(index=True, name=None):
        cells = "".join(table_cell(value) for value in observations)
        rows.append(f'<tr><th scope="row">{escape(label)}</th>{cells}</tr>')
    title = escape(f"Показатели · {province_labels[province]}")
    return (
        f'<div class="sheet-scroll" tabindex="0" role="region" aria-label="{title}">'
        f'<table class="annual" aria-label="{title}" style="--other-cols:{len(table.columns)}">'
        f'<colgroup><col class="province-col"><col span="{len(table.columns)}"></colgroup>'
        f'<thead><tr><th scope="col">Показатель</th>{years}</tr></thead>'
        f'<tbody>{"".join(rows)}</tbody></table></div>'
    )


# Индексы в порядке блоков, затем сводный индекс. Строки Median из Excel
# не являются наблюдениями провинций и не входят в annual_series.
balance_indices = pd.concat({
    label: annual_series(f"I_Блок{number}") for label, number in block_choices
} | {"Сводный индекс": annual_series(overview)}, axis=1)
balance_years = sorted(
    (year for year in balance_indices.index.get_level_values("год").unique() if 2011 <= year <= 2024),
    reverse=True,
)


def balance_view(year):
    if year is None:
        return "", ""
    indices = balance_indices.xs(int(year), level="год").reindex(names["Русский"].tolist())
    medians = indices.median(axis=0, skipna=True)
    delta = indices.subtract(medians, axis="columns")
    distances = delta.abs()
    closest = distances.eq(distances.min(axis=0), axis="columns")
    # Для сопоставимости оценка требует полных данных по всем восьми блокам.
    blocks = delta.drop(columns="Сводный индекс")
    scores = blocks.pow(2).sum(axis=1, min_count=len(blocks.columns)).dropna()
    if scores.empty:
        status = f"**{year}** · Данных по всем восьми блокам для определения эталона недостаточно."
    else:
        winners = ", ".join(province_labels[p] for p in scores.index[scores.eq(scores.min())])
        score_text = format(scores.min(), ".2f").replace(".", ",")
        status = (f"**{year} · Эталон по восьми блокам — {winners}.** "
                  f"Сумма квадратов отклонений {score_text} · В сравнении {len(scores)} из {len(indices)} провинций.")

    header = "".join(f'<th scope="col">{escape(label)}</th>' for label in indices.columns)
    rows = []
    for province, row in delta.iterrows():
        cells = []
        for label, value in row.items():
            if pd.isna(value):
                cells.append('<td title="Нет данных">—</td>')
                continue
            text = format(value, "+.2f" if value else ".2f").replace(".", ",")
            if closest.at[province, label]:
                text = f"<strong>{text}</strong>"
            cells.append(f'<td title="{str(value).replace(".", ",")}">{text}</td>')
        rows.append(f'<tr><th scope="row">{escape(province_labels[province])}</th>{"".join(cells)}</tr>')
    footer = "".join(table_cell(value) if pd.notna(value) else "<td>—</td>" for value in medians)
    table = (
        f'<div class="sheet-scroll" tabindex="0"><table aria-label="Отклонения от медиан за {year} год">'
        f'<thead><tr><th scope="col">Провинция</th>{header}</tr></thead>'
        f'<tbody>{"".join(rows)}</tbody>'
        f'<tfoot><tr><th scope="row">Медиана</th>{footer}</tr></tfoot></table></div>'
    )
    return status, table


# Позиции заданы для шкалы от худшего результата (0) к лучшему (1).
palette = [(0, "#df2029"), (0.24, "#f27524"), (0.50, "#ffbd38"),
           (0.72, "#f6e852"), (0.78, "#b6d94d"), (1, "#08753b")]

def map_years(matrix):
    return [str(y) for y in matrix.columns if 2011 <= y <= 2024]


period_hint = "Выберите год или введите период, например 2015-2020, и нажмите Enter."


def period_bounds(text, available):
    match = re.fullmatch(r"([0-9]{4})(?:\s*[-–—]\s*([0-9]{4}))?", str(text).strip())
    if match:
        first = int(match[1])
        last = int(match[2] or match[1])
        if 2011 <= first <= last <= 2024:
            if all(str(y) in available for y in range(first, last + 1)):
                return first, last
            raise ValueError("На выбранном листе отсутствует год из указанного периода.")
    raise ValueError("Неверный период. Укажите год или интервал от 2011 до 2024, например 2015-2020. Начальный год должен быть не позже конечного.")


def draw_focus(figure, focus):
    pid = ru_to_id[focus]
    geometry = next(f["geometry"] for f in boundaries["features"] if f["id"] == pid)
    polygons = geometry["coordinates"]
    if geometry["type"] == "Polygon":
        polygons = [polygons]

    # Разрывы отделяют острова и внутренние кольца друг от друга.
    lon, lat = [], []
    for polygon in polygons:
        for ring in polygon:
            for x, y in ring:
                lon.append(x)
                lat.append(y)
            lon.append(None)
            lat.append(None)

    figure.add_scattergeo(
        lon=lon, lat=lat, mode="lines", line=dict(color="#182433", width=2.8),
        fill="none", connectgaps=False, hoverinfo="skip", showlegend=False,
    )
    # Подпись у северной границы основного контура оставляет заливку видимой.
    main_ring = max((polygon[0] for polygon in polygons), key=len)
    x, y = max(main_ring, key=lambda point: point[1])
    figure.add_scattergeo(
        lon=[x], lat=[y], mode="text", text=[focus], textposition="top center",
        textfont=dict(size=14, weight=700, color="#182433",
                      shadow="0 0 2px white, 0 0 4px white, 0 0 6px white"),
        hoverinfo="skip", showlegend=False,
    )


map_titles = {key: label for label, key in map_options}


def configure_map(figure, focus, subject, period, mode, version=None):
    if focus is not None:
        draw_focus(figure, focus)
    title_lines = wrap(subject, width=54)
    subtitle = " · ".join(part for part in (period, mode, version) if part)
    title = "<br>".join(escape(line) for line in title_lines)
    title += f'<br><span style="font-size:13px">{escape(subtitle)}</span>'
    filename = " — ".join(part for part in (period, subject, version, mode) if part)
    filename = re.sub(r'[<>:"/\\|?*]', "_", filename)

    figure.update_geos(visible=False, fitbounds="locations",
                       projection=dict(type="conic equal area", parallels=[25, 47],
                                       rotation=dict(lon=105)))
    heading_height = 51 + 22 * len(title_lines)
    figure.update_layout(
        title=dict(text=title, x=0.015, xanchor="left", y=0.95, yref="container", yanchor="top", font=dict(size=16)),
        height=540 + heading_height, margin=dict(l=8, r=12, t=heading_height, b=8),
        meta=dict(export_filename=filename), uirevision="china", paper_bgcolor="white",
    )


CLUSTERS = "Кластеры по соседству"
cluster_colors = {
    "HH": "#168447", "HL": "#f3d334", "LH": "#f28522", "LL": "#cf3036",
    "Без кластера": "#aeb8c3", "Вне выборки": "#e8ebef",
}
cluster_labels = {overview: "Сводный индекс"}
cluster_labels.update({f"I_Блок{number}": label for label, number in block_choices})
# Короткие имена блоков в названиях листов Морана.
moran_blocks = {
    "Безоп": 1, "Здрав": 2, "ОбразНаука": 3, "СоцПодд": 4, "СпортКульт": 5,
    "Транспорт": 6, "Экология": 7, "ЭконРазв": 8,
}
moran_sheets = {key: {} for key in cluster_labels}
for book, sheet in sheet_keys:
    match = re.fullmatch(r"Moran_Соседство_(?:([^_]+)_)?([0-9]{4})", sheet)
    if book == MAIN and match:
        key = f"I_Блок{moran_blocks[match[1]]}" if match[1] else overview
        moran_sheets[key][match[2]] = sheet

map_headers = ["Провинция", "Территория", "Значение", "Место", "Разница с выбранной"]
cluster_headers = ["Провинция", "Территория", "Тип кластера", "Значение индекса", "Локальный I Морана"]


@lru_cache(maxsize=None)
def moran_data(sheet):
    frame = values[MAIN, sheet]
    header = frame.index[frame["J"].eq("Тип кластера") & frame["A"].eq("Province")].item()
    rows = frame.loc[header + 1:header + len(province_ids), ["A", "B", "H", "I", "J"]].copy()
    rows.columns = ["province", "index_value", "included", "local_i", "cluster"]
    rows["province"] = rows["province"].map(to_ru).map(ru_to_id)
    if set(rows["province"]) != set(province_ids):
        raise ValueError(f"Проверьте названия провинций на листе {sheet}.")
    rows.loc[rows["included"].ne(1) | rows["cluster"].isin(["", None]), "cluster"] = "Без кластера"
    rows["cluster"] = rows["cluster"].fillna("Без кластера")
    if not rows["cluster"].isin(["HH", "HL", "LH", "LL", "Без кластера"]).all():
        raise ValueError(f"Неизвестный тип кластера на листе {sheet}.")
    return rows.set_index("province")


def cluster_view(scope, year, focus):
    if scope is None:
        return "Выберите блок или сводный индекс.", None, []
    data = moran_data(moran_sheets[scope][year])
    categories = data["cluster"]
    figure = go.Figure()
    for group, color in cluster_colors.items():
        features = [f for f in boundaries["features"] if categories.get(f["id"], "Вне выборки") == group]
        if not features:
            continue
        locations = [f["id"] for f in features]
        figure.add_choropleth(
            name=group, showlegend=True, showscale=False,
            geojson={"type": "FeatureCollection", "features": features},
            locations=locations, z=[0] * len(locations), zmin=0, zmax=1,
            colorscale=[[0, color], [1, color]], marker_line_color="white", marker_line_width=0.8,
            text=[to_ru.get(pid, pid) for pid in locations],
            hovertemplate=f"<b>%{{text}}</b> · {year}<br>{group}<extra></extra>",
        )
    configure_map(figure, focus, map_titles[scope], year, CLUSTERS)
    figure.update_layout(legend=dict(title_text="Тип кластера", itemclick=False, itemdoubleclick=False))
    status = f"**{cluster_labels[scope]} · {year}**"
    if focus is None:
        return status, figure, []
    origin = ru_to_id[focus]
    neighbours = adj.columns[adj.loc[origin].eq(1)]
    rows = []
    for pid in [origin, *neighbours]:
        record = data.loc[pid]
        rows.append([to_ru[pid], "Выбранная" if pid == origin else "Соседняя", record["cluster"],
                     cell_text(record["index_value"]), cell_text(record["local_i"])])
    return status, figure, rows


def map_view(selection, version, year, mode, focus):
    number = int(selection) if selection.isdecimal() else None
    if number is None:
        sheet = selection
        options = [RANK, FIXED, CLUSTERS]
    else:
        sheet = indicator_sheets[number][version][1]
        options = [RANK, CLUSTERS]
    if mode not in options:
        mode = RANK

    matrix = province_years(sheet)
    years = map_years(matrix)
    year_cfg = gr.update(choices=years, value=year, allow_custom_value=True, info=period_hint)
    mode_cfg = gr.update(choices=options, value=mode, interactive=len(options) > 1)
    version_cfg = gr.update(visible=number is not None)
    try:
        first, last = period_bounds(year, years)
    except ValueError as exc:
        gr.Warning(str(exc))
        return version_cfg, year_cfg, mode_cfg, str(exc), None, []

    span = list(range(first, last + 1))
    period_label = str(first) if first == last else f"{first}–{last}"
    value_label = "Значение" if first == last else "Среднее за период"
    observations = matrix.loc[:, span].replace([np.inf, -np.inf], np.nan)
    # Если в интервале есть пропуск, среднего за полный период нет.
    data = observations.mean(axis=1, skipna=False).rename(index=ru_to_id).reindex(province_ids)
    lower_is_better = False
    if number is not None and version == RAW:
        # Направление именно этого показателя берём из формулы нормирования МНСК.
        formula = formulas[indicator_sheets[number][NORMAL]].at[5, "B"]
        sign = re.search(r"100\s*([+-])", str(formula))
        if not sign:
            raise ValueError(f"Проверьте направление нормирования показателя №{number}.")
        lower_is_better = sign[1] == "-"
    ranks = data.rank(ascending=lower_is_better, method="min")
    count = int(data.count())
    if mode == FIXED:
        scores = (data - 1) / 99
    elif count and ranks.max() > 1:
        scores = 1 - (ranks - 1) / (ranks.max() - 1)
    else:
        scores = data.where(data.isna(), 1.0)

    figure = go.Figure()
    # Пропуски и территории вне выборки также рисуем, чтобы сохранялся контур карты.
    for group in ("значение", "нет данных", "вне выборки"):
        features, z, tips = [], [], []
        for feature in boundaries["features"]:
            pid = feature["id"]
            actual = ("вне выборки" if pid not in data.index else
                      "нет данных" if pd.isna(data[pid]) else "значение")
            if actual != group:
                continue
            city = to_ru.get(pid, pid)
            if actual == "значение":
                tip = (f"<b>{city}</b> · {period_label}<br>{value_label}: {cell_text(float(data[pid]))}"
                       f"<br>Место: {int(ranks[pid])} из {count}")
                z.append(float(scores[pid]))
            else:
                tip = f"<b>{city}</b> · {period_label}<br>{actual.capitalize()}"
                z.append(0)
            tips.append(tip)
            features.append(feature)
        if not features:
            continue
        plain = "#bcc4ce" if group == "нет данных" else "#e8ebef"
        if mode == FIXED:
            ticks, labels = [0, 0.5, 1], ["1", "50,5", "100"]
        elif count and ranks.max() > 1:
            ticks, labels = [0, 1], [str(int(ranks.max())), "1"]
        else:
            ticks, labels = [1], ["1"]
        figure.add_choropleth(
            geojson={"type": "FeatureCollection", "features": features},
            locations=[f["id"] for f in features], z=z, zmin=0, zmax=1,
            colorscale=palette if group == "значение" else [[0, plain], [1, plain]],
            text=tips, hovertemplate="%{text}<extra></extra>",
            marker_line_color="white", marker_line_width=0.8,
            showscale=(group == "значение"),
            colorbar=dict(title="Индекс" if mode == FIXED else "Место",
                          tickvals=ticks, ticktext=labels, len=0.55, thickness=14),
        )
    subject = map_titles[selection]
    if number is not None:
        subject = f"Показатель {subject}"
    configure_map(figure, focus, subject, period_label, mode, version if number is not None else None)
    direction = "меньшее значение лучше" if lower_is_better else "большее значение лучше"
    status = (f"**{sheet} · {period_label}** · {value_label}  ·  Данные для {count} из 31 территорий"
              f"  ·  {direction}. Серый — нет данных, светло-серый — вне выборки.")
    if focus is None:
        return version_cfg, year_cfg, mode_cfg, status, figure, []
    origin = ru_to_id[focus]
    neighbours = adj.columns[adj.loc[origin].eq(1)].tolist()
    table = []
    for pid in [origin, *neighbours]:
        value = data[pid]
        difference = value - data[origin]
        table.append([to_ru[pid], "Выбранная" if pid == origin else "Соседняя",
                      cell_text(float(value)) if pd.notna(value) else "Нет данных",
                      str(int(ranks[pid])) if pd.notna(value) else "—",
                      cell_text(float(difference)) if pd.notna(difference) else "—"])
    return version_cfg, year_cfg, mode_cfg, status, figure, table


# Повторный запуск ячейки заменяет ранее открытый интерфейс.
if "mnsk_viewer" in globals():
    mnsk_viewer.close()

with gr.Blocks(head=map_head, css=css, title="Таблицы МНСК", analytics_enabled=False) as mnsk_viewer:
    gr.Markdown("# Таблицы МНСК")

    with gr.Tab("Таблицы"):
        category_box = gr.Dropdown(
            choices=[INDICATOR, *groups],
            value=INDICATOR,
            label="Что посмотреть",
        )

        block_box = gr.Dropdown(
            label="Блок", choices=block_choices, value=None, interactive=True,
        )
        with gr.Row():
            indicator_box = gr.Dropdown(
                label="Показатель", choices=[], value=None, interactive=False, scale=3,
            )
            version_box = gr.Radio(
                choices=[RAW, NORMAL],
                value=RAW,
                label="Вид показателя",
                interactive=False,
            )

        sheet_box = gr.Dropdown(
            choices=[],
            value=None,
            label="Книга и лист",
            visible=False,
            interactive=True,
        )

        output = gr.HTML("Выберите блок.", elem_id="sheet-output")

        gr.on(
            triggers=[
                category_box.input,
                block_box.input,
                indicator_box.input,
                version_box.input,
                sheet_box.input,
            ],
            fn=select_table,
            inputs=[category_box, block_box, indicator_box, version_box, sheet_box],
            outputs=[block_box, indicator_box, version_box, sheet_box, output],
            trigger_mode="always_last",
        )

    with gr.Tab("Карта"):
        with gr.Row():
            selection = gr.Dropdown(map_options, value=overview, label="Показатель или индекс",
                                    filterable=True, interactive=True)
            cluster_scope = gr.Dropdown(
                [(label, key) for key, label in cluster_labels.items()], value=None,
                label="Блок или сводный индекс", interactive=True, visible=False,
            )
            map_version = gr.Radio([RAW, NORMAL], value=RAW, visible=False, label="Вид показателя")
            initial_years = map_years(province_years(overview))
            initial_year = "2019" if "2019" in initial_years else max(initial_years)
            year = gr.Dropdown(
                initial_years, value=initial_year, label="Год", allow_custom_value=True,
                interactive=True, info=period_hint,
            )
        mode = gr.Radio([RANK, FIXED, CLUSTERS], value=RANK, label="Цвет провинции")
        initial = map_view(overview, RAW, initial_year, RANK, "Пекин")
        status = gr.Markdown(initial[3])
        chart = gr.Plot(value=initial[4], show_label=False, elem_id="china-map")
        focus = gr.Dropdown(
            sorted(ru_to_id), value="Пекин", interactive=True,
            label="Провинция и её сухопутные соседи",
        )
        with gr.Row():
            clear_focus = gr.Button("Удалить выделение", elem_id="clear-province-focus")
            download_map = gr.Button("Скачать изображение")

        # Используем экспорт самого отображённого графика вместе с его заголовком и легендой.
        download_map.click(fn=None, inputs=[], outputs=[], js="""
        () => {
            const plot = document.querySelector("#china-map .js-plotly-plot");
            const camera = plot?._fullLayout?._modeBar?.buttons.flat().find(b => b.name === "toImage");
            if (!camera || !plot.layout.meta?.export_filename) {
                throw new Error("Карта ещё не готова к скачиванию.");
            }
            Object.assign(plot._context.toImageButtonOptions, {
                format: "png", filename: plot.layout.meta.export_filename, scale: 2,
                width: Math.max(1000, Math.round(plot.getBoundingClientRect().width)),
                height: Math.round(plot.getBoundingClientRect().height)
            });
            camera.click(plot);
            return [];
        }
        """)
        neighbours = gr.Dataframe(
            headers=map_headers,
            value=initial[5], datatype="str", interactive=False,
        )
        clicked = gr.Textbox(visible="hidden", interactive=True, elem_id="province-click")

        def update_map(selection_value, version_value, year_value, mode_value,
                       province, clicked_id, scope, event: gr.EventData):
            by_click = event.target is clicked
            clearing = getattr(event.target, "elem_id", None) == clear_focus.elem_id
            if clearing:
                province = None
            elif by_click:
                clicked_id = clicked_id.partition("|")[0]
                if clicked_id not in province_ids:
                    return (gr.skip(),) * 11
                province = to_ru[clicked_id]

            clustered = mode_value == CLUSTERS
            if clustered:
                years = sorted(moran_sheets[scope or overview])
                if year_value not in years:
                    year_value = years[-1]
                info, fig, rows = cluster_view(scope, year_value, province)
                headers = cluster_headers
                version_cfg = gr.update(visible=False)
                year_cfg = gr.update(choices=years, value=year_value, allow_custom_value=False,
                                     info="Год расчёта кластеров по соседству.")
                mode_cfg = gr.skip()
            else:
                version_cfg, year_cfg, mode_cfg, info, fig, rows = map_view(
                    selection_value, version_value, year_value, mode_value, province
                )
                headers = map_headers

            table_cfg = gr.update(headers=headers, value=rows)
            focus_cfg = gr.update(value=province) if by_click or clearing else gr.skip()
            return (version_cfg, year_cfg, mode_cfg, info, fig, table_cfg, focus_cfg,
                    gr.update(visible=not clustered), gr.update(visible=clustered),
                    gr.update(interactive=province is not None), gr.update(interactive=fig is not None))

        controls = [selection, map_version, year, mode, focus, clicked, cluster_scope]
        gr.on(
            [widget.input for widget in controls] + [clear_focus.click],
            fn=update_map, inputs=controls,
            outputs=[map_version, year, mode, status, chart, neighbours, focus, selection, cluster_scope,
                     clear_focus, download_map],
            trigger_mode="always_last",
        )

        mnsk_viewer.load(fn=None, js="""
        () => {
            let clickNumber = 0;
            function sendProvince(event) {
                const id = event.points?.[0]?.location;
                const field = document.querySelector("#province-click textarea, #province-click input");
                if (!id || !field) return;
                field.value = id + "|" + (++clickNumber);
                field.dispatchEvent(new Event("input", {bubbles: true}));
            }

            // Plotly может создать новый график после смены года или вкладки.
            document.addEventListener("pointerdown", event => {
                const plot = event.target.closest("#china-map .js-plotly-plot");
                if (!plot || typeof plot.on !== "function") return;
                plot.removeListener("plotly_click", sendProvince);
                plot.on("plotly_click", sendProvince);
            }, true);
        }
        """)

    with gr.Tab("Провинции"):
        province_box = gr.Dropdown(
            choices=province_choices, value=province_choices[0][1],
            label="Провинция", filterable=True, interactive=True,
        )
        province_output = gr.HTML(province_table(province_choices[0][1]), elem_id="province-output")
        province_box.input(
            fn=province_table, inputs=province_box, outputs=province_output, trigger_mode="always_last",
        )

    with gr.Tab("Баланс"):
        balance_year = gr.Dropdown([str(y) for y in balance_years], value=None,
                                   label="Год", interactive=True)
        gr.Markdown(
            "В строках провинций показано отклонение в пунктах: индекс минус медиана за выбранный год. "
            "Жирным выделены ближайшие к медиане значения каждого столбца. "
            "В нижней строке приведены медианы исходных индексов. Прочерк означает отсутствие данных. "
            "Эталон года определяется по наименьшей сумме квадратов отклонений восьми блоков."
        )
        balance_status = gr.Markdown()
        balance_output = gr.HTML(elem_id="balance-output")
        balance_year.input(balance_view, inputs=balance_year, outputs=[balance_status, balance_output],
                           trigger_mode="always_last")

mnsk_viewer.launch(
    server_name="127.0.0.1",
    share=False,
    inline=False,
    inbrowser=True,
)